# ⚡ Duino API — Google Colab

**Cloud-agnostic hyperscale AI inference — Gemma 4 · Streaming · React UI**

> 💡 **Before running:** Go to `Runtime → Change runtime type → T4 GPU`

---

## 📦 Cell 1 — Clone & Install
_Run once. ~2 min on first run._

In [ ]:
import os, sys

REPO_URL = 'https://github.com/jalalmansour/Duino_API.git'
REPO_DIR = '/content/Duino_API'

# ── Clone ─────────────────────────────────────────────────────────────────────
if not os.path.exists(REPO_DIR):
    print('Cloning Duino API...')
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo exists — pulling latest...')
    !git -C {REPO_DIR} pull --quiet

# ── CRITICAL: register repo on Python path BEFORE any imports ─────────────────
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

# ── Install via pip directly (works on Colab — no python3.10 -m pip issue) ────
print('Installing core requirements...')
!pip install -r {REPO_DIR}/requirements.txt -q
!pip install -e {REPO_DIR} -q
print('✅ Installation complete')

## 🔑 Cell 2 — (Optional) Set Tokens
_Use Colab Secrets (🔑 sidebar) instead of pasting here._

In [ ]:
import os

# ── Method A: Colab Secrets (recommended — click the 🔑 icon in the sidebar) ──
# Add HF_TOKEN and NGROK_AUTHTOKEN there. They load automatically.

# ── Method B: Paste inline (not recommended for shared notebooks) ──────────────
# os.environ['HF_TOKEN']        = 'hf_xxx'   # HuggingFace (needed for Gemma 4)
# os.environ['NGROK_AUTHTOKEN'] = 'xxx'       # Free at https://ngrok.com

# ── Status ────────────────────────────────────────────────────────────────────
print(f"HF_TOKEN set     : {'✅' if os.environ.get('HF_TOKEN')        else '⚠️  not set'}")
print(f"NGROK_AUTHTOKEN  : {'✅' if os.environ.get('NGROK_AUTHTOKEN') else '⚠️  not set (Colab proxy used instead)'}")

## 🚀 Cell 3 — Start Platform
_Installs inference deps, loads model, starts API + UI, creates HTTPS URL._

In [ ]:
import sys, os

REPO_DIR = '/content/Duino_API'

# Ensure path is set (in case kernel restarted)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

from studio.backend.colab import start

result = start(
    model    = 'gemma-4-2b',   # 'gemma-4-9b' or 'gemma-4-27b' if more VRAM
    api_port = 8000,
    ui_port  = 3000,
    expose   = True,           # creates public HTTPS URL
)

api_url    = result['api_url']
ui_url     = result['ui_url']
api_key    = result['api_key']
embed_html = result['embed_html']

print(f'\n📡 API  : {api_url}')
print(f'🎨 UI   : {ui_url}')
print(f'🔑 Key  : {api_key}')
print(f'📖 Docs : {api_url}/docs')

## 🎨 Cell 4 — Embed React UI
_Interactive chat UI rendered inside this notebook cell._

In [ ]:
# Method A — Colab native port proxy (most reliable)
from google.colab import output as colab_output
colab_output.serve_kernel_port_as_iframe(3000, height=700, width='100%')

In [ ]:
# Method B — direct iframe with postMessage config injection
from IPython.display import HTML, display
display(HTML(embed_html))

## 🧪 Cell 5 — Test API

In [ ]:
import requests

resp = requests.post(
    f'{api_url}/v1/chat/completions',
    headers={'X-API-Key': api_key},
    json={
        'model': 'gemma-4-2b',
        'messages': [{'role': 'user', 'content': 'What is the capital of France? One sentence.'}],
        'max_tokens': 64,
    },
)
print('Status:', resp.status_code)
print('Answer:', resp.json()['choices'][0]['message']['content'])
print('Tokens:', resp.json()['usage'])

In [ ]:
# ── Streaming test ────────────────────────────────────────────────────────────
import requests, json

print('Streaming response:')
with requests.post(
    f'{api_url}/v1/chat/completions',
    headers={'X-API-Key': api_key},
    json={'model': 'gemma-4-2b',
          'messages': [{'role': 'user', 'content': 'Write a short poem about AI.'}],
          'max_tokens': 128, 'stream': True},
    stream=True,
) as r:
    for line in r.iter_lines():
        if line:
            line = line.decode()
            if line.startswith('data: ') and line[6:] != '[DONE]':
                delta = json.loads(line[6:]).get('choices',[{}])[0].get('delta',{}).get('content','')
                print(delta, end='', flush=True)
print()

## 💬 Cell 6 — Multi-turn Session Chat

In [ ]:
import requests

# Create session
sess = requests.post(f'{api_url}/v1/sessions',
    headers={'X-API-Key': api_key},
    json={'model_id': 'gemma-4-2b'}).json()
session_id = sess['session_id']
print(f'Session: {session_id}')

def chat(message):
    r = requests.post(f'{api_url}/v1/chat/completions',
        headers={'X-API-Key': api_key},
        json={'model': 'gemma-4-2b',
              'messages': [{'role': 'user', 'content': message}],
              'session_id': session_id, 'max_tokens': 256}).json()
    return r['choices'][0]['message']['content']

print('\nUser: My name is Jalal.')
print('AI  :', chat('My name is Jalal.'))
print('\nUser: What is my name?')
print('AI  :', chat('What is my name?'))  # Should remember

## ⚛️ Cell 7 — Run Any React Project Inside Colab
_Scaffold a Vite + React app, start the dev server, open as iframe._

In [ ]:
import subprocess, threading, time, os
from google.colab import output as colab_output

REACT_PORT = 4000
REACT_DIR  = '/content/my-react-app'

if not os.path.exists(REACT_DIR):
    print('Scaffolding Vite + React...')
    !npm create vite@latest {REACT_DIR} -- --template react -y 2>/dev/null || \
     npm create vite@latest {REACT_DIR} -- --template react
    !npm install --prefix {REACT_DIR} --silent
    print('✅ React app ready')
else:
    print('✅ App exists')

# Start Vite dev server
def _vite():
    subprocess.run(['npm', 'run', 'dev', '--', '--host', '0.0.0.0',
                    '--port', str(REACT_PORT)], cwd=REACT_DIR)

threading.Thread(target=_vite, daemon=True).start()
time.sleep(6)
print(f'Vite running on port {REACT_PORT}')

colab_output.serve_kernel_port_as_iframe(REACT_PORT, height=600, width='100%')

In [ ]:
# ── Next.js inside Colab ──────────────────────────────────────────────────────
import subprocess, threading, time, os
from google.colab import output as colab_output

NEXT_PORT = 4001
NEXT_DIR  = '/content/my-next-app'

if not os.path.exists(NEXT_DIR):
    !npx create-next-app@latest {NEXT_DIR} --ts --eslint --no-git --yes

def _next():
    subprocess.run(['npm', 'run', 'dev', '--', '-p', str(NEXT_PORT)], cwd=NEXT_DIR)

threading.Thread(target=_next, daemon=True).start()
time.sleep(8)

colab_output.serve_kernel_port_as_iframe(NEXT_PORT, height=600, width='100%')

## 📊 Cell 8 — Health & Models

In [ ]:
import requests

health = requests.get(f'{api_url}/v1/health').json()
models = requests.get(f'{api_url}/v1/models', headers={'X-API-Key': api_key}).json()

print('\n📊 Platform Health')
print('=' * 40)
for k, v in health.items():
    print(f'  {k:<22} : {v}')

print('\n🤖 Available Models')
print('=' * 40)
for m in models.get('data', []):
    print(f"  - {m['id']}")

## ♾️ Cell 9 — Keep Alive

In [ ]:
import time

print('Platform running — keeping session alive. Press STOP (■) to terminate.')
print(f'  API  : {api_url}')
print(f'  UI   : {ui_url}')
print(f'  Docs : {api_url}/docs')
print()

for i in range(10_000):
    time.sleep(300)
    print(f'= [{i+1} × 5min alive]', end='', flush=True)